In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from sqlalchemy import create_engine

ROOT           = Path.cwd().parent
DATA_RAW       = ROOT / 'data' / 'raw'
DATA_PROCESSED = ROOT / 'data' / 'processed'
SQL_PATH       = ROOT / 'sql' / 'queries.sql'

# Load dataframes
master  = pd.read_csv(DATA_PROCESSED / 'master.csv')
orders  = pd.read_csv(DATA_RAW / 'olist_orders_dataset.csv')
reviews = pd.read_csv(DATA_RAW / 'olist_order_reviews_dataset.csv')

# Parse dates
master['order_purchase_timestamp'] = pd.to_datetime(master['order_purchase_timestamp'])
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])

print(f"master:  {master.shape[0]:,} rows")
print(f"orders:  {orders.shape[0]:,} rows")
print(f"reviews: {reviews.shape[0]:,} rows")

master:  110,197 rows
orders:  99,441 rows
reviews: 99,224 rows


In [9]:
# Connect to SQLite
engine = create_engine(f'sqlite:///{DATA_PROCESSED}/olist.db')

# Push tables into database
master.to_sql('master',   engine, if_exists='replace', index=False)
orders.to_sql('orders',   engine, if_exists='replace', index=False)
reviews.to_sql('reviews', engine, if_exists='replace', index=False)

print('Database created at:', DATA_PROCESSED / 'olist.db')

Database created at: /Users/zunmyathsu/Desktop/ecommerce-analysis/data/processed/olist.db


In [10]:
def load_query(name):
    text  = SQL_PATH.read_text()
    start = text.find(f'-- [{name}]')
    end   = text.find('-- [end]', start)
    if start == -1:
        raise ValueError(f"Query '{name}' not found in queries.sql")
    sql = text[start:end].split('\n', 1)[1].strip().rstrip(';')
    return sql

print("Ready — use load_query('query_name') to run queries")

Ready — use load_query('query_name') to run queries


In [16]:
df_monthly = pd.read_sql(load_query('monthly_revenue'), engine)
df_monthly

,order_month,monthly_revenue,order_count,revenue_change,pct_change
0,2016-09,143.46,1,NaN,NaN
1,2016-10,46490.66,265,46347.20,32306.7
2,2016-12,19.62,1,-46471.04,-100.0
3,2017-01,127482.37,750,127462.75,649657.2
4,2017-02,271239.32,1653,143756.95,112.8
5,2017-03,414330.95,2546,143091.63,52.8
6,2017-04,390812.40,2303,-23518.55,-5.7
7,2017-05,566851.40,3546,176039.00,45.0
8,2017-06,490050.37,3135,-76801.03,-13.5
9,2017-07,566299.08,3872,76248.71,15.6
